# Notebook 08 — WWF RGB orthomosaics | Musanga V3_1

**Primary objective:** evaluate **native RGB quality and resolution**, and select orthomosaics for **new exhaustive Musanga annotations** across a broad forest regeneration/disturbance gradient. This is **not** a cross-validation experiment with the old UAV sites.

**Workflow:** A Setup → B GCS inventory → C survey map → D RGB native-resolution audit → E small-window visual QC → F diverse annotation site screening.

**Confirmed pilot:** `UTM33S_Plot135_RGB.tif` is EPSG:32733, **10 cm GSD**, 3 bands, 13 × 3 km and **has no overviews**. A 650-pixel preview of the *entire raster* may force huge reads. **Inspect bounded native-pixel crops instead.**

All GCS operations are read-only. No large mosaic downloads, new annotation labels or modifications to existing V3_1 model data.

## A — Environment, authentication and workspace
Install dependencies and connect Colab to Google Drive / GCS. Earth Engine is optional for the RGB-first objective.

In [ ]:
# 00 — Install dependencies
%pip -q install google-cloud-storage geopandas shapely pyogrio rasterio folium earthengine-api geemap fsspec gcsfs

In [ ]:
# 01 — Authenticate Google Cloud, mount Drive and initialize Earth Engine
from google.colab import auth, drive
from pathlib import Path
import os
auth.authenticate_user()
drive.mount('/content/drive', force_remount=False)
# The registered project used in the existing Musanga satellite notebooks.
EE_PROJECT = (os.environ.get('EE_PROJECT') or 'musanga-511110').strip()
import ee
def initialize_earth_engine():
    """Initialize with Musanga cloud project; authenticate interactively if necessary."""
    try:
        ee.Initialize(project=EE_PROJECT)
    except Exception as first_error:
        print('Earth Engine needs authentication:', str(first_error)[:350])
        ee.Authenticate()
        ee.Initialize(project=EE_PROJECT)
    # Lightweight server read verifies the project actually works.
    print('Earth Engine connected:', EE_PROJECT,
          '| server check:', ee.Number(1).getInfo())
    return True
EE_READY=False
try:
    EE_READY=initialize_earth_engine()
except Exception as err:
    print('Earth Engine setup did not complete:',repr(err))
    print('The GCS inventory and RGB tests work without EE. For TMF, fix the project or authenticate before H2.')
MUSANGA_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/Musanga')
WORK_DIR = MUSANGA_ROOT/'V3_1'/'NB08_WWF_Lidar'
WORK_DIR.mkdir(parents=True,exist_ok=True)
print('Outputs:',WORK_DIR)

## B — WWF dataset inventory
Inspect bucket object sizes, filenames and matched per-plot RGB, LiDAR and CHM coverage. Metadata only.

In [ ]:
# 02 — Bucket inventory (metadata only)
from google.cloud import storage
import pandas as pd, re
from pathlib import PurePosixPath
BUCKET='carbon-maps'
client=storage.Client()
rows=[]
for b in client.list_blobs(BUCKET):
    if b.name.endswith('/'): continue
    m=re.search(r'(UTM\d{1,2}[NS])[_/-]?(?:Plot|plot)[_-]?(\d+)',b.name,re.I)
    plot_id=f'{m.group(1).upper()}_Plot{int(m.group(2))}' if m else None
    rows.append(dict(path=b.name,extension=PurePosixPath(b.name).suffix.lower(),size_GB=(b.size or 0)/1e9,plot_id=plot_id,updated=str(b.updated)))
df=pd.DataFrame(rows)
df.to_csv(WORK_DIR/'carbon_maps_inventory.csv',index=False)
print('Files:',len(df),'GB:',round(df.size_GB.sum(),1))
display(df.groupby('extension').agg(files=('path','size'),GB=('size_GB','sum')).sort_values('GB',ascending=False).head(20))

In [ ]:
# 03 — Match WWF LiDAR products to plot IDs
patterns={
 'RGB':r'^001_Lidar_Rasters/RGB_mosaic/.*_RGB\.tif$',
 'CHM':r'^001_Lidar_Rasters/Lidar_2meter/CHM/.*\.tif$',
 'DTM':r'^001_Lidar_Rasters/Lidar_2meter/DTM/.*\.tif$',
 'DSM':r'^001_Lidar_Rasters/Lidar_2meter/DSM/.*\.tif$',
 'LAS':r'^00-raw-data/Plots/.*\.las$',
 'ECW':r'^00-raw-data/Plots/.*\.ecw$'}
products=pd.DataFrame({'plot_id':sorted(df.plot_id.dropna().unique())})
for kind,pattern in patterns.items():
    counts=df.loc[df.path.str.contains(pattern,case=False,regex=True,na=False)].groupby('plot_id').size()
    products[kind+'_n']=products.plot_id.map(counts).fillna(0).astype(int)
display(products.head(20))
print('Plots with RGB:',(products.RGB_n>0).sum())
products.to_csv(WORK_DIR/'wwf_plot_product_inventory.csv',index=False)

## C — WWF survey geography
Load the 216 georeferenced plot footprints and display a reliable offline-first survey map. Esri satellite is optional; OSM 403 errors are avoided.

In [ ]:
# 04 — Download only small plot GIS sidecars; do not read broken Ferry_Lines geometry
import tempfile
import geopandas as gpd
from pathlib import Path
prefix='003_GIS_Layers/Lidar_Plots/'
local=Path(tempfile.mkdtemp(prefix='wwf_gis_'))
for b in client.list_blobs(BUCKET,prefix=prefix):
    if b.name.lower().endswith(('.shp','.shx','.dbf','.prj','.cpg','.qpj')):
        b.download_to_filename(str(local/Path(b.name).name))
shps=list(local.glob('*.shp'))
assert shps, 'No plot shapefile found'
plots=gpd.read_file(shps[0]).to_crs('EPSG:4326')
plots=plots[plots.geometry.notna() & ~plots.geometry.is_empty].copy()
plots['geometry']=plots.geometry.make_valid()
plots['plot_id']=plots.Name.astype(str).str.strip()
plots['utm_zone']=plots.plot_id.str.extract(r'^(UTM\d{1,2}[NS])',expand=False)
plots=plots.merge(products,on='plot_id',how='left',validate='one_to_one')
print('Features:',len(plots),'Unique plots:',plots.plot_id.nunique())
display(plots.groupby('utm_zone').agg(plots=('plot_id','size'),rgb=('RGB_n',lambda x:(x>0).sum())))
plots.to_file(WORK_DIR/'wwf_lidar_plots.gpkg',driver='GPKG')

In [ ]:
# 05 — Robust interactive map (NO OpenStreetMap requests)
# The previous OSM basemap returned HTTP 403 within Colab.
# Esri satellite is optional; the default offline map always works.
import folium
from folium.plugins import Fullscreen, MiniMap
from IPython.display import display
from html import escape
import pandas as pd

# tiles=None ensures ZERO requests to restricted OSM tile servers.
m = folium.Map(location=[-2.5, 23.5], zoom_start=6, tiles=None,
               control_scale=True, prefer_canvas=True)

# Reliable fallback: blank geographic background (no external tile dependency).
folium.TileLayer(
    tiles='data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAQAAAC1HAwCAAAAC0lEQVR42mP8/x8AAusB9Y9UxQAAAABJRU5ErkJggg==',
    attr='Local blank geographic background', name='Offline (always works)',
    overlay=False, show=True, control=True, max_zoom=19
).add_to(m)

# Optional imagery; switch it on only if your Colab browser can access Esri.
folium.TileLayer(
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
    attr='Tiles © Esri', name='Esri World Imagery (optional)',
    overlay=False, show=False, control=True, max_zoom=19
).add_to(m)

colors={'UTM33S':'#9467bd','UTM34S':'#e3771c','UTM34N':'#2ca02c',
        'UTM35S':'#d62728','UTM35N':'#1f77b4','UTM36N':'#8c564b'}

# Plot geometry gives survey context even if ALL basemap servers fail.
foot=folium.FeatureGroup(name='Survey footprints',show=True)
folium.GeoJson(
    plots[['plot_id','geometry']].to_json(),
    style_function=lambda f: {'color':'#555555','weight':1.5,'fillOpacity':0.08},
    tooltip=folium.GeoJsonTooltip(fields=['plot_id'])
).add_to(foot)
foot.add_to(m)

for zone,group in plots.groupby('utm_zone'):
    layer=folium.FeatureGroup(name=f'{zone}: {len(group)} plots',show=True)
    for _,r in group.iterrows():
        pt=r.geometry.representative_point()
        counts={k:int(r[k+'_n']) if pd.notna(r[k+'_n']) else 0
                for k in ('RGB','LAS','CHM','DTM','DSM')}
        label=escape(str(r.plot_id))
        html='<b>'+label+'</b><br>'+'<br>'.join(f'{k}: {v}' for k,v in counts.items())
        col=colors.get(zone,'gray')
        folium.CircleMarker(
            [pt.y,pt.x],radius=5,color=col,weight=2,fill=True,
            fill_color=col if counts['RGB'] else '#ffffff',fill_opacity=.9,
            tooltip=label,popup=folium.Popup(html,max_width=300)
        ).add_to(layer)
    layer.add_to(m)

# Geographic graticule: geographic reference even with the offline background.
graticule=folium.FeatureGroup(name='Latitude / longitude grid',show=True)
for latitude in range(-12,7,2):
    folium.PolyLine([[latitude,12],[latitude,32]],color='#a0a0a0',
                    weight=.6,opacity=.4,tooltip=f'{latitude}° latitude').add_to(graticule)
for longitude in range(12,33,2):
    folium.PolyLine([[-13,longitude],[7,longitude]],color='#a0a0a0',
                    weight=.6,opacity=.4,tooltip=f'{longitude}° longitude').add_to(graticule)
graticule.add_to(m)

w,s,e,n=plots.total_bounds
m.fit_bounds([[s,w],[n,e]])
Fullscreen().add_to(m)
folium.LayerControl(collapsed=False).add_to(m)

output=WORK_DIR/'wwf_lidar_survey_map.html'
m.save(str(output))
print('Offline-capable map:',output)
print('To see satellite imagery: select Esri World Imagery in the layer control.')
display(m)

### C.1 — Optional Musanga UAV ROIs (context only)
Their spatial overlap is **not** required. Run this cell only if you want to see the original UAV sites on the map.

In [ ]:
# 05A — Overlay the real Musanga V3_1 UAV ROI polygons on the WWF map
# Source of truth: V3_1 audited manifest and its ROI.shp (no guessed coordinates).
from pathlib import Path
from shapely.geometry import box
import geopandas as gpd
import rasterio
import folium
import pandas as pd
from IPython.display import display
import json
from html import escape

drive_root=Path('/content/drive/MyDrive')
final_root=next((q for q in [
    drive_root/'Colab Notebooks/Musanga/Final',
    drive_root/'Musanga/Final'] if q.is_dir()), None)
assert final_root, 'Musanga/Final not found. Mount Drive and check V3_1 paths.'
manifest_file=final_root/'Outputs/V3_1/00_input_audit/00_input_manifest.json'
assert manifest_file.exists(), (
    'Approved V3_1 input manifest missing: '+str(manifest_file)
    +'. Run V3_1 Notebook 00 / Block 10 first.')
manifest=json.loads(manifest_file.read_text())
roi_file=Path(manifest['roi'])
assert roi_file.is_file(), f'ROI shapefile missing: {roi_file}'
uav_roi=gpd.read_file(roi_file)
assert uav_roi.crs is not None, 'ROI.shp has no CRS; refusing to guess.'
uav_roi=uav_roi[uav_roi.geometry.notna() & ~uav_roi.geometry.is_empty].copy()
uav_roi.geometry=uav_roi.geometry.make_valid()
uav_roi=uav_roi[uav_roi.geometry.notna() & ~uav_roi.geometry.is_empty].copy()
assert len(uav_roi), 'No valid UAV ROI geometries.'

# Assign ROI polygons to a site by physical overlap with real mosaic footprints.
# Intersections are measured in the ROI's native metric CRS, not lat/lon degrees.
mosaic_records=[]
for entry in manifest['mosaics']:
    image_file=Path(entry['path'])
    if not image_file.exists():
        print('Mosaic missing; cannot assign this footprint:',image_file)
        continue
    with rasterio.open(image_file) as ds:
        assert ds.crs is not None, f'No CRS: {image_file}'
        g=gpd.GeoSeries([box(*ds.bounds)],crs=ds.crs).to_crs(uav_roi.crs).iloc[0]
        mosaic_records.append(dict(site=str(entry['site']),file=str(entry['file']),geometry=g))
assert mosaic_records, 'No readable UAV mosaics for ROI attribution.'

# Aggregate overlapping mosaics by forest site, avoiding duplicate MOS geometry.
from shapely.ops import unary_union
site_footprints={
    site:unary_union([x['geometry'] for x in mosaic_records if x['site']==site])
    for site in sorted(set(x['site'] for x in mosaic_records))}
roi_rows=[]
for index,row in uav_roi.iterrows():
    g=row.geometry
    overlaps={site:float(g.intersection(foot).area)
              for site,foot in site_footprints.items()}
    best=max(overlaps,key=overlaps.get)
    if overlaps[best]<=0:
        best='Unassigned'
    roi_rows.append(best)
uav_roi['site']=roi_rows
uav_roi['roi_row']=[int(i) for i in range(len(uav_roi))]
print('UAV ROI features:',len(uav_roi))
print('Site assignment:',uav_roi.site.value_counts().to_dict())
if (uav_roi.site=='Unassigned').any():
    print('WARNING: Unassigned ROI polygons are shown separately, not silently removed.')

roi_wgs=uav_roi.to_crs('EPSG:4326')
roi_colors={'Baego':'#ef4444','MOS':'#f59e0b','Yangambi':'#10b981',
            'Yoko':'#ec4899','Unassigned':'#111827'}
for site, group in roi_wgs.groupby('site'):
    layer=folium.FeatureGroup(name=f'UAV ROI — {site} ({len(group)})',show=True)
    color=roi_colors.get(site,'#111827')
    folium.GeoJson(
        group[['site','roi_row','geometry']].to_json(),
        style_function=lambda feat,c=color:{
            'color':c,'weight':3.5,'fillColor':c,'fillOpacity':.16},
        highlight_function=lambda feat:{'weight':5,'fillOpacity':.3},
        tooltip=folium.GeoJsonTooltip(fields=['site','roi_row'],
            aliases=['Musanga site','ROI row'])
    ).add_to(layer)
    # Distinct labels at true ROI interiors
    for _,rr in group.iterrows():
        pt=rr.geometry.representative_point()
        folium.Marker([pt.y,pt.x],
            icon=folium.DivIcon(html=(
                '<div style="font: bold 11px sans-serif;white-space:nowrap;'
                'text-shadow:1px 1px 2px white,-1px -1px 2px white;color:'
                +color+';">'+escape(site)+'</div>'))
        ).add_to(layer)
    layer.add_to(m)

# Drop old layer controls before adding the revised one.
for key, child in list(m._children.items()):
    if isinstance(child,folium.map.LayerControl):
        del m._children[key]
folium.LayerControl(collapsed=False).add_to(m)

# Plot boundaries remain visible against the complete WWF footprint.
allbounds=pd.concat([
    pd.Series([plots.total_bounds[0],roi_wgs.total_bounds[0]]),
    pd.Series([plots.total_bounds[1],roi_wgs.total_bounds[1]]),
    pd.Series([plots.total_bounds[2],roi_wgs.total_bounds[2]]),
    pd.Series([plots.total_bounds[3],roi_wgs.total_bounds[3]])
],axis=1)
m.fit_bounds([[min(plots.total_bounds[1],roi_wgs.total_bounds[1]),
               min(plots.total_bounds[0],roi_wgs.total_bounds[0])],
              [max(plots.total_bounds[3],roi_wgs.total_bounds[3]),
               max(plots.total_bounds[2],roi_wgs.total_bounds[2])]])

uav_roi.to_file(WORK_DIR/'musanga_v3_1_roi_wwf_overlay.gpkg',driver='GPKG')
map_path=WORK_DIR/'wwf_lidar_plus_musanga_uav_roi.html'
m.save(str(map_path))
print('Saved map:',map_path)
print('Saved georeferenced ROI:',WORK_DIR/'musanga_v3_1_roi_wwf_overlay.gpkg')
display(m)

## D — RGB orthomosaic native-resolution audit
Measure pixel size **from raster transforms**, not nominal filenames. Report footprint dimensions, band count, CRS, overviews, block layout and color interpretation. Avoid full-resolution reads.

In [ ]:
# D1 — Inspect a single RGB mosaic's native metadata (cheap remote header reads)
import rasterio
import pandas as pd
from rasterio.crs import CRS
from pyproj import Geod
sample_path='001_Lidar_Rasters/RGB_mosaic/UTM33S_Plot135_RGB.tif'
uri='/vsigs/'+BUCKET+'/'+sample_path
with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR', CPL_VSIL_CURL_ALLOWED_EXTENSIONS='.tif'):
    with rasterio.open(uri) as src:
        print('File:',sample_path)
        print('CRS:',src.crs)
        print('Width × height:',src.width,'×',src.height)
        print('Bands:',src.count, 'Dtypes:',src.dtypes, 'Color interpretation:',[v.name for v in src.colorinterp])
        print('Pixel size (native CRS units):',src.res)
        print('GSD in cm if CRS projected in metres:',tuple(round(100*abs(v),2) for v in src.res) if src.crs and src.crs.is_projected and src.crs.linear_units=='metre' else 'requires CRS unit conversion')
        print('Extent:',src.bounds)
        print('Internal overviews:',src.overviews(1))
        print('Storage layout:',src.profile.get('tiled'), 'Block shapes:',src.block_shapes)
        print('Compression:',src.profile.get('compress'),'Nodata:',src.nodata)
        print('Pixel count:',f'{src.width*src.height:,}')
print('NOTE: Missing overviews — NEVER downsample entire full-resolution TIFF for an overview.')

### D.2 — Metadata audit across multiple RGB mosaics
Start with **6 mosaics from different UTM zones** (modifiable), not all 216. Remote metadata access can still incur GCS requests. This saves a QC table with resolution in centimetres and flags missing overviews.

In [ ]:
# D2 — Sampled metadata audit: no image pixel reads
import math, pandas as pd, rasterio
MAX_RGB_METADATA=6
# One example per UTM zone; choose more later when working
rgb_paths=(df[df.path.str.fullmatch(r'001_Lidar_Rasters/RGB_mosaic/.*_RGB\.tif',case=False,na=False)]
           .copy())
rgb_paths['utm']=rgb_paths.plot_id.str.extract(r'^(UTM\d{1,2}[NS])',expand=False)
pick=rgb_paths.groupby('utm',sort=True).head(1).head(MAX_RGB_METADATA)
metadata_rows=[]
for rec in pick.itertuples(index=False):
    row=dict(plot_id=rec.plot_id,path=rec.path,size_GB=rec.size_GB)
    try:
        with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR'):
            with rasterio.open('/vsigs/'+BUCKET+'/'+rec.path) as s:
                row.update(crs=str(s.crs),width=s.width,height=s.height,
                           bands=s.count,gsd_x_native=s.res[0],gsd_y_native=s.res[1],
                           overviews=len(s.overviews(1)),tiled=s.profile.get('tiled'),
                           compression=s.profile.get('compress'),
                           block_x=s.block_shapes[0][1],block_y=s.block_shapes[0][0])
                if s.crs and s.crs.is_projected and s.crs.linear_units=='metre':
                    row.update(gsd_cm_x=100*abs(s.res[0]),gsd_cm_y=100*abs(s.res[1]))
                else:
                    row.update(gsd_cm_x=float('nan'),gsd_cm_y=float('nan'))
    except Exception as e:
        row['error']=str(e)[:250]
    metadata_rows.append(row)
meta=pd.DataFrame(metadata_rows)
meta.to_csv(WORK_DIR/'wwf_rgb_metadata_audit.csv',index=False)
display(meta)
print('Saved:',WORK_DIR/'wwf_rgb_metadata_audit.csv')

## E — RGB quality inspection at **native resolution**
The main annotation decision requires individual tree crown visibility. **Do not use `src.read(out_shape=(3,650,650))` on an entire 13 km mosaic without overviews.** Instead, read **small native-pixel windows** (40 × 40 m at 10 cm = 400 × 400 pixels). Inspect a few different locations within each mosaic; the default is 4 small windows on the first pilot mosaic.

Note: a large striped TIFF can cause some extra data transfer even for a small window; inspect block shapes and monitor network usage.

In [ ]:
# E1 — RGB diagnostic: valid-data windows, true colours and native pixels
# Important: zero-valued margins are common in orthomosaics, even without nodata metadata.
import numpy as np
import matplotlib.pyplot as plt
import rasterio
from rasterio.windows import Window
from pathlib import Path
from rasterio.plot import reshape_as_image

QC_PLOT_ID = 'UTM33S_Plot135'
QC_WINDOW_METRES = 40
QC_MAX_WINDOWS = 4
QC_MAX_PROBES = 30  # bounded window reads only; no full-mosaic scan
QC_VALID_THRESHOLD = 0.80  # require at least 80% nonblack pixels
# RGB/BGR ambiguity is diagnosed by displaying both on the first selected patch.
source_rows = df[df.path.str.fullmatch(
    r'001_Lidar_Rasters/RGB_mosaic/' + QC_PLOT_ID + r'_RGB\.tif',
    case=False, na=False)]
assert len(source_rows) == 1, f'Expected exactly 1 RGB mosaic, got {len(source_rows)}'
qc_path = source_rows.iloc[0].path
preview_dir = WORK_DIR / 'RGB_QC_native_windows_v2'
preview_dir.mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(8108)

with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR'):
    with rasterio.open('/vsigs/' + BUCKET + '/' + qc_path) as src:
        assert src.crs and src.crs.is_projected and src.crs.linear_units == 'metre'
        assert src.count >= 3
        W = min(src.width, max(1, round(QC_WINDOW_METRES / abs(src.res[0]))))
        H = min(src.height, max(1, round(QC_WINDOW_METRES / abs(src.res[1]))))
        print('Raster:',qc_path)
        print('Native GSD (cm):',100*abs(src.res[0]),100*abs(src.res[1]))
        print('Band colors:',[c.name for c in src.colorinterp])
        print('Bands dtype:',src.dtypes,'Nodata:',src.nodata)
        print('Block shapes:',src.block_shapes,'Overviews:',src.overviews(1))
        print('Scanning up to',QC_MAX_PROBES,'small windows for nonblack coverage...')
        candidates = [(0.5,0.5),(0.35,0.5),(0.65,0.5),
                      (0.5,0.35),(0.5,0.65),
                      (0.25,0.25),(0.75,0.25),(0.25,0.75),(0.75,0.75)]
        candidates += list(map(tuple,rng.uniform(0.1,0.9,(QC_MAX_PROBES,2))))
        keep = []
        checked = []
        for fx, fy in candidates[:QC_MAX_PROBES]:
            col = int(np.clip(round(fx*src.width-W/2),0,src.width-W))
            row = int(np.clip(round(fy*src.height-H/2),0,src.height-H))
            win = Window(col,row,W,H)
            a = src.read(indexes=[1,2,3],window=win,masked=True)
            # A pixel is usable only if all channels are valid AND not all zeros.
            data = np.ma.filled(a,0)
            valid_mask = (~np.ma.getmaskarray(a).any(axis=0)) & np.any(data != 0,axis=0)
            good = float(valid_mask.mean())
            checked.append((col,row,round(good,3)))
            if good >= QC_VALID_THRESHOLD:
                keep.append((win,data,valid_mask,good))
                if len(keep) >= QC_MAX_WINDOWS: break
        print('Window diagnostics (col,row,valid fraction):',checked)
        if not keep:
            print('No windows meet minimum coverage. Could be sparse flight footprint or nodata encoding.')
        for i,(win,data,valid,good) in enumerate(keep,1):
            # Native integer channels: retain relative R/G/B levels (no separate 2-98% per-band stretch).
            img = np.moveaxis(data.astype('float32'),0,-1)
            vals = img[valid]
            if np.issubdtype(np.dtype(src.dtypes[0]),np.integer):
                scale=float(np.iinfo(np.dtype(src.dtypes[0])).max)
                natural=np.clip(img/scale,0,1)
            else:
                # Shared stretch across all channels; preserves relative band colours.
                hi=np.percentile(vals,99) if vals.size else 1
                natural=np.clip(img/max(hi,1e-6),0,1)
            natural[~valid] = 0
            bgr=natural[:,:,::-1]
            if i==1:
                fig,axs=plt.subplots(1,2,figsize=(12,6))
                axs[0].imshow(natural,interpolation='nearest')
                axs[0].set_title('Original band order (1,2,3)')
                axs[1].imshow(bgr,interpolation='nearest')
                axs[1].set_title('Reversed order (3,2,1) — diagnostic only')
                for ax in axs:ax.axis('off')
                plt.show();plt.close(fig)
            fig,ax=plt.subplots(figsize=(7,7))
            ax.imshow(natural,interpolation='nearest')
            ax.set_title(f'{QC_PLOT_ID} | native {100*abs(src.res[0]):g} cm | valid {good:.1%} | window {i}')
            ax.axis('off')
            output=preview_dir/f'{QC_PLOT_ID}_window{i}.png'
            fig.savefig(output,dpi=160,bbox_inches='tight')
            plt.show();plt.close(fig)
            print('Saved:',output,'pixel window:',win)
print('Judge crown delineation ONLY using valid imagery and verified band order.')


### E.2 — Manual visual quality score
Record whether crowns can be delineated, not merely nominal pixel size. Evaluate blur, shadows, mosaicking seams, color balance and the visibility of small pioneer crowns. Save a simple screening table for each inspected sample.

In [ ]:
# E2 — Editable QC sheet (fill after examining true-resolution windows)
qc_form=pd.DataFrame([{
    'plot_id':'UTM33S_Plot135','gsd_cm':10.0,
    'crown_visibility_1to5':None,'sharpness_1to5':None,
    'shadow_problem_1to5':None,'seam_problem_1to5':None,
    'possible_musanga_visible':'UNCERTAIN',
    'forest_structure':'UNREVIEWED',
    'exhaustive_annotation_feasible':'UNREVIEWED',
    'notes':'Manual review required; no species inference from GSD alone'
}])
qc_form.to_csv(WORK_DIR/'wwf_rgb_visual_qc_template.csv',index=False)
display(qc_form)
print('QC template:',WORK_DIR/'wwf_rgb_visual_qc_template.csv')

### E.3 — Six-mosaic RGB atlas for Musanga annotation feasibility
Test a deliberately small sample across UTM zones. For each mosaic, inspect up to three **35 × 35 m native-resolution crops** (with at most 12 spatial probes). Export both original band order and reversed order, because the blue/yellow colours might reflect BGR interpretation. Results include a contact sheet, individual PNGs and a CSV containing exact raster-window bounds. This is a **visual screening step**, not a validated Musanga identification or forest age label. Some mosaics may have no valid crop among the few sampled positions: record that explicitly, do not label them poor-quality. The first run makes at most 72 small window reads; large striped TIFFs may still incur extra I/O.

In [ ]:
# E3 — RGB annotation feasibility atlas: bounded native-resolution crops
# Output: 6 plots × 3 examples, individual PNGs, contact sheet, QC table.
# Zero full-mosaic reads and zero full TIFF downloads.
import numpy as np, pandas as pd, matplotlib.pyplot as plt, rasterio
from rasterio.windows import Window, bounds as window_bounds
from pathlib import Path
from PIL import Image
from IPython.display import display
import warnings

ATLAS_PLOTS = [
    'UTM33S_Plot135','UTM34N_Plot105','UTM34S_Plot60',
    'UTM35N_Plot100','UTM35S_Plot79','UTM36N_Plot104'
]
ATLAS_CROP_M = 35                # 350 px if GSD is 10 cm
ATLAS_EXAMPLES_PER_PLOT = 3
ATLAS_MAX_PROBES_PER_PLOT = 12  # cap remote reads if flight footprint has holes
ATLAS_MIN_VALID = 0.70
# NOTE: plot-wide 1-99% stretches are not used; the 8-bit original bands are retained.
atlas_dir = WORK_DIR/'RGB_annotation_feasibility_atlas'
atlas_dir.mkdir(parents=True,exist_ok=True)

def make_display(raw, mask, dtype):
    """Both original and reversed RGB channels; identical scaling in both."""
    array = np.moveaxis(raw.astype(np.float32), 0, -1)
    good = array[mask]
    if np.issubdtype(np.dtype(dtype), np.integer):
        max_value=float(np.iinfo(np.dtype(dtype)).max)
        original=np.clip(array/max_value,0,1)
    else:
        hi=float(np.nanpercentile(good,99.5)) if good.size else 1.
        original=np.clip(array/max(hi,1e-8),0,1)
    original[~mask]=0
    return original, original[:,:,::-1]

rows=[]
images=[]
for plot_id in ATLAS_PLOTS:
    selection=df[df.path.eq(f'001_Lidar_Rasters/RGB_mosaic/{plot_id}_RGB.tif')]
    if len(selection)!=1:
        print('SKIP: missing mosaic',plot_id)
        continue
    uri='/vsigs/'+BUCKET+'/'+selection.iloc[0].path
    print('\nOPEN',plot_id,flush=True)
    try:
        with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR'):
            with rasterio.open(uri) as ds:
                assert ds.crs and ds.crs.is_projected and ds.crs.linear_units=='metre'
                assert ds.count>=3
                nx=min(ds.width,max(1,int(round(ATLAS_CROP_M/abs(ds.res[0])))))
                ny=min(ds.height,max(1,int(round(ATLAS_CROP_M/abs(ds.res[1])))))
                print('GSD (cm):',tuple(round(abs(r)*100,2) for r in ds.res),
                      '| bands:',[c.name for c in ds.colorinterp],
                      '| overviews:',ds.overviews(1),'| blocks:',ds.block_shapes[0],flush=True)
                # Deterministic staggered probes, including central transect;
                # avoids sampling only the corners of a narrow flight swath.
                locations=[(.5,.5),(.25,.5),(.75,.5),(.5,.25),(.5,.75),
                           (.35,.35),(.65,.65),(.35,.65),(.65,.35),
                           (.15,.5),(.85,.5),(.5,.85)]
                n_saved=0
                for k,(fx,fy) in enumerate(locations[:ATLAS_MAX_PROBES_PER_PLOT]):
                    left=int(np.clip(round(fx*ds.width-nx/2),0,ds.width-nx))
                    top=int(np.clip(round(fy*ds.height-ny/2),0,ds.height-ny))
                    win=Window(left,top,nx,ny)
                    ma=ds.read(indexes=[1,2,3],window=win,masked=True)
                    raw=np.ma.filled(ma,0)
                    good=(~np.ma.getmaskarray(ma).any(axis=0)) & np.any(raw!=0,axis=0)
                    fraction=float(good.mean())
                    if fraction<ATLAS_MIN_VALID:
                        continue
                    original,reversed_rgb=make_display(raw,good,ds.dtypes[0])
                    n_saved+=1
                    rgb=(original*255).round().astype('uint8')
                    bgr=(reversed_rgb*255).round().astype('uint8')
                    stem=f'{plot_id}_crop{n_saved:02d}'
                    Image.fromarray(rgb).save(atlas_dir/f'{stem}_bands123.png')
                    Image.fromarray(bgr).save(atlas_dir/f'{stem}_bands321.png')
                    # Actual window bounding box for relocation; in source CRS
                    x0,y0,x1,y1=window_bounds(win,ds.transform)
                    rows.append(dict(plot_id=plot_id,crop=n_saved,probe=k+1,
                         x_min=x0,y_min=y0,x_max=x1,y_max=y1,
                         crs=str(ds.crs),gsd_cm_x=abs(ds.res[0])*100,
                         gsd_cm_y=abs(ds.res[1])*100,
                         valid_fraction=fraction,channel_order='UNVERIFIED',
                         crown_delineation='UNREVIEWED',musanga_identifiable='UNREVIEWED',
                         forest_environment='UNREVIEWED',notes=''))
                    images.append((stem,original,reversed_rgb,fraction))
                    print(f'  crop {n_saved}: probe {k+1}, valid {fraction:.1%}',flush=True)
                    if n_saved>=ATLAS_EXAMPLES_PER_PLOT:break
                if n_saved==0:
                    print('  WARNING: no valid crop among probed positions; not proof of bad imagery')
    except Exception as exc:
        print('ERROR inspecting',plot_id,type(exc).__name__,str(exc)[:400],flush=True)

if rows:
    table=pd.DataFrame(rows)
    table.to_csv(atlas_dir/'atlas_annotation_QC.csv',index=False)
    display(table[['plot_id','crop','gsd_cm_x','valid_fraction','channel_order',
                   'crown_delineation','musanga_identifiable']])
    # Contact sheet at modest display size only; native PNGs are retained.
    fig,axes=plt.subplots(len(images),2,figsize=(9,3.8*len(images)),squeeze=False)
    for j,(stem,a,b,valid) in enumerate(images):
        for col,img in enumerate((a,b)):
            axes[j,col].imshow(img,interpolation='nearest')
            axes[j,col].set_title(f'{stem} | '+('bands 1/2/3' if col==0 else 'bands 3/2/1'),fontsize=9)
            axes[j,col].axis('off')
    fig.suptitle('WWF RGB — native crops, identical channel scaling (colour order unverified)',fontsize=13)
    fig.tight_layout()
    fig.savefig(atlas_dir/'atlas_contact_sheet.jpg',dpi=110,bbox_inches='tight')
    display(fig);plt.close(fig)
    print('Saved contact sheet, individual native crops and QC:',atlas_dir)
else:
    print('No valid crop produced. Check nodata encoding and file layout; do NOT infer that RGB is bad.')


## F — Screening mosaics for new exhaustive annotations
Choose spatially dispersed candidate RGB mosaics as a starting list. **Spatial dispersion is not a proxy for forest age.** Classification into young secondary forest, regrowth, disturbed forest, mature canopy or open land requires image review and external land-cover history.

In [ ]:
# 08 — Catalogue RGB mosaics and select a spatially dispersed screening set
import numpy as np, pandas as pd
from pathlib import PurePosixPath
rgb=df[df.path.str.fullmatch(r'001_Lidar_Rasters/RGB_mosaic/.*_RGB\.tif',case=False,na=False)].copy()
geo=plots.copy()
cent=geo.geometry.representative_point()
geo['longitude']=cent.x;geo['latitude']=cent.y
catalog=geo[['plot_id','utm_zone','longitude','latitude','geometry']].drop(columns='geometry').merge(
    rgb[['plot_id','path','size_GB']].rename(columns={'path':'rgb_gcs_path','size_GB':'rgb_GB'}),
    how='inner',on='plot_id',validate='one_to_one')
catalog['visual_forest_class']='UNREVIEWED'
catalog['musanga_presence']='UNREVIEWED'
catalog['annotation_suitability']='UNREVIEWED'
catalog['notes']=''
catalog.to_csv(WORK_DIR/'wwf_rgb_annotation_catalog.csv',index=False)
print('RGB candidates:',len(catalog))
display(catalog.groupby('utm_zone').agg(n=('plot_id','size'),GB=('rgb_GB','sum')))
# Screen a geographically dispersed subset across all UTM zones.
# Farthest-point sampling on geographic coordinates within zones; not interpreted as age.
from pyproj import Geod
geod=Geod(ellps='WGS84')
chosen=[]
for zone, group in catalog.groupby('utm_zone'):
    target=min(5,len(group))
    pool=group.reset_index(drop=True)
    selected=[0]
    while len(selected)<target:
        distances=[]
        for j,rr in pool.iterrows():
            if j in selected: distances.append(-1);continue
            mind=min(geod.inv(rr.longitude,rr.latitude,
                              pool.iloc[k].longitude,pool.iloc[k].latitude)[2]
                     for k in selected)
            distances.append(mind)
        selected.append(int(np.argmax(distances)))
    chosen.append(pool.iloc[selected])
screen=pd.concat(chosen,ignore_index=True)
screen['review_priority']='INITIAL_SCREEN'
screen.to_csv(WORK_DIR/'wwf_rgb_initial_screen.csv',index=False)
print('Initial screening set:',len(screen),'(spatially dispersed, NOT preclassified)')
display(screen[['plot_id','utm_zone','latitude','longitude','rgb_GB']])

## G — Next decisions
Compare native GSD and observed RGB quality across zones; shortlist mosaics with visually annotatable crowns and a balanced gradient of forest environments, including Musanga-free exhaustive negatives. Expand the screening set only after reviewing the pilot true-resolution crops.

### G.1 — TMF landscape descriptors for **all WWF LiDAR strips** (2014–2015)

**Purpose:** characterize the **complete 216-strip survey**, before selecting further 125 m annotation circles. This uses the actual WWF survey footprints, not the six pilot strips, and is fully independent of UAV sites. The JRC TMF 30 m annual classification supplies *landscape-scale* evidence: proportions of undisturbed, degraded, deforested, regrowing, water, and other land cover in 2014 and 2015; fraction of the 2014–2015 landscape that changed class; and a conservative indicator of clearing already present in 2014 following forest in 2011/2012. The acquisition year is only known as 2014–2015, so no later-year conditions enter sampling descriptors.

**Interpretation:** class fractions describe historical TMF conditions **within the mapped transect polygon**, which is not necessarily the exact valid 10 cm RGB coverage. Deforested ≠ bare clearing, and regrowth ≠ known Musanga. A simple diversity index helps flag mixed landscapes; it is not a biological diversity measurement. All output values are computed server-side on Earth Engine, downloaded as small aggregated tables, then ranked and plotted. No large WWF imagery files are downloaded.

This cell can run after **C / cell 04** (once `plots` and `WORK_DIR` exist). It does **not** require H.1, H.2 or annotation circles.

In [ ]:
# G.1 — WWF LIDAR STRIP TMF GRADIENT AUDIT (ALL 216 FOOTPRINTS)
# 2014/2015 composition, recent clearing, class-change, landscape diversity
import ee, geopandas as gpd, pandas as pd, numpy as np, matplotlib.pyplot as plt
from shapely.geometry import mapping
from pathlib import Path
from IPython.display import display
import math, time

EE_PROJECT = (globals().get('EE_PROJECT') or 'musanga-511110').strip()
try:
    ee.Initialize(project=EE_PROJECT)
    ee.Number(1).getInfo()
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)
print('Earth Engine:', EE_PROJECT)

assert 'plots' in globals(), 'Run Section C / cell 04 to load WWF footprints'
assert 'plot_id' in plots.columns and plots.plot_id.is_unique
assert plots.crs is not None

TMF_STRIP_DIR = WORK_DIR / 'TMF_strip_descriptors'
TMF_STRIP_DIR.mkdir(parents=True,exist_ok=True)
BATCH_SIZE = 12    # small requests; keeps Colab responsive and allows resume
RECOMPUTE = False  # false: reuse completed batch CSVs on rerun
EE_SCALE_M = 30
LABELS = ['undisturbed','degraded','deforested','regrowth','water','other']
TMF = ee.ImageCollection('projects/JRC/TMF/v1_2025/AnnualChanges').mosaic()
a11, a12, a14, a15 = [TMF.select('Dec'+str(y)) for y in [2011,2012,2014,2015]]
area_ha = ee.Image.pixelArea().divide(10000)
bands = [area_ha.rename('footprint_ha')]
for year, img in [(2014,a14),(2015,a15)]:
    for value, name in enumerate(LABELS, 1):
        bands.append(area_ha.multiply(img.eq(value).unmask(0)).rename(
            f'ha_{year}_{name}'))
# Describes pixels already deforested by 2014 that were forest in 2011 or 2012.
forest_pre = (a11.eq(1).Or(a11.eq(2))
              .Or(a12.eq(1)).Or(a12.eq(2)))
recent_clear = a14.eq(3).And(forest_pre).And(a15.eq(3))
bands += [
    area_ha.multiply(recent_clear.unmask(0)).rename('ha_recent_clear_pre2014'),
    area_ha.multiply(a14.neq(a15).And(a14.mask()).And(a15.mask()).unmask(0))
       .rename('ha_changed_class_2014_2015'),
]
descriptor_image = ee.Image.cat(bands).toFloat()

geo = plots[['plot_id','utm_zone','geometry']].copy().to_crs('EPSG:4326')
geo = geo.loc[geo.geometry.notna() & ~geo.geometry.is_empty].sort_values('plot_id')
# Use compact coordinates; spatial analysis is carried out using the original polygons.
print('Strips:',len(geo), '| EE scale:',EE_SCALE_M,'m')
partial_files = []
for start in range(0,len(geo),BATCH_SIZE):
    batch=geo.iloc[start:start+BATCH_SIZE]
    outfile=TMF_STRIP_DIR/f'tmf_strip_batch_{start:03d}_{start+len(batch):03d}.csv'
    partial_files.append(outfile)
    if outfile.exists() and not RECOMPUTE:
        print('Cached:',outfile.name)
        continue
    features=[]
    for rec in batch.itertuples(index=False):
        features.append(ee.Feature(
            ee.Geometry(mapping(rec.geometry),proj='EPSG:4326',geodesic=False),
            {'plot_id':str(rec.plot_id),'utm_zone':str(rec.utm_zone)}))
    collection=ee.FeatureCollection(features)
    result=descriptor_image.reduceRegions(
        collection=collection,
        reducer=ee.Reducer.sum(),
        scale=EE_SCALE_M,
        tileScale=4
    )
    raw=result.getInfo()['features']
    records=[{k:v for k,v in f['properties'].items()} for f in raw]
    out=pd.DataFrame(records)
    assert len(out)==len(batch),f'Unexpected row count: {len(out)} vs {len(batch)}'
    out.to_csv(outfile,index=False)
    print(f'Calculated {start+1}–{start+len(batch)} / {len(geo)}')

desc=pd.concat([pd.read_csv(p) for p in partial_files],ignore_index=True)
assert len(desc)==len(geo) and desc.plot_id.nunique()==len(geo), (
    'Missing or duplicate WWF strip summaries')
numeric=[c for c in desc if c.startswith('ha_') or c=='footprint_ha']
desc[numeric]=desc[numeric].apply(pd.to_numeric,errors='coerce')
for year in (2014,2015):
    for name in LABELS:
        desc[f'pct_{year}_{name}']=100*desc[f'ha_{year}_{name}']/desc.footprint_ha
    desc[f'pct_{year}_TMF_forest']=(
        desc[f'pct_{year}_undisturbed']+
        desc[f'pct_{year}_degraded']+
        desc[f'pct_{year}_regrowth'])
    desc[f'pct_{year}_disturbed_or_regrowth']=(
        desc[f'pct_{year}_degraded']+
        desc[f'pct_{year}_regrowth'])
desc['pct_recent_clear_pre2014']=100*desc.ha_recent_clear_pre2014/desc.footprint_ha
desc['pct_changed_class_2014_2015']=100*desc.ha_changed_class_2014_2015/desc.footprint_ha
# Normalized Shannon diversity of the 6-class landscape (not biological diversity).
p=desc[[f'pct_2014_{x}' for x in LABELS]].fillna(0).to_numpy()/100.
p=np.clip(p,0,1)
desc['landscape_diversity_2014']=-(np.where(p>0,p*np.log(np.maximum(p,1e-12)),0)
                                   .sum(axis=1))/np.log(len(LABELS))
desc['max_class_share_2014_pct']=100*p.max(axis=1)
# Ranking emphasizes disturbed / regrowing contexts, but retains entire 6-class landscape.
desc['gradient_priority_score']=(
    desc.pct_2014_degraded*0.25+
    desc.pct_2014_regrowth*0.40+
    desc.pct_recent_clear_pre2014*0.20+
    desc.landscape_diversity_2014*100*0.15
)
desc=desc.sort_values('gradient_priority_score',ascending=False).reset_index(drop=True)
desc['tmf_priority_rank']=np.arange(1,len(desc)+1)
out_csv=TMF_STRIP_DIR/'WWF_all_strips_TMF_2014_2015_descriptors.csv'
desc.to_csv(out_csv,index=False)
joined=geo.merge(desc,on=['plot_id','utm_zone'],validate='one_to_one')
out_gpkg=TMF_STRIP_DIR/'WWF_all_strips_TMF_2014_2015_descriptors.gpkg'
joined.to_file(out_gpkg,driver='GPKG')
print('Saved:',out_csv)
print('Saved:',out_gpkg)
important=['plot_id','utm_zone','footprint_ha','pct_2014_undisturbed',
           'pct_2014_degraded','pct_2014_deforested','pct_2014_regrowth',
           'pct_2014_other','pct_recent_clear_pre2014',
           'pct_changed_class_2014_2015','landscape_diversity_2014',
           'tmf_priority_rank']
display(desc[important].head(25).round(2))
print('\nOverall across-strip variability, percentages (not area weighted):')
display(desc[[c for c in important if c.startswith('pct_')]].describe(
    percentiles=[.1,.25,.5,.75,.9]).round(1))
print('\nMean composition by WWF UTM zone:')
display(desc.groupby('utm_zone')[[f'pct_2014_{x}' for x in LABELS]].mean().round(1))
# Readable stacked composition of the top 30 diversified/disturbed strips.
top=desc.head(30).iloc[::-1]
fig,axs=plt.subplots(1,2,figsize=(16,11),gridspec_kw={'width_ratios':[3,2]})
left=np.zeros(len(top))
palette=['#215b45','#c5a642','#de6b56','#80b779','#4d86b0','#bbb9b4']
for label,color in zip(LABELS,palette):
    v=top[f'pct_2014_{label}'].fillna(0).to_numpy()
    axs[0].barh(top.plot_id,v,left=left,label=label,color=color)
    left+=v
axs[0].set_xlim(0,100)
axs[0].set_xlabel('TMF class area within strip (%) — 2014')
axs[0].set_title('Top 30 strips ranked by recovery / disturbance diversity')
axs[0].legend(ncol=3,fontsize=9,loc='lower center',bbox_to_anchor=(.5,-.14))
axs[1].scatter(desc.pct_2014_undisturbed,desc.pct_2014_regrowth,
               s=20+desc.pct_recent_clear_pre2014*2,
               c=desc.pct_2014_degraded,cmap='viridis',vmin=0,
               vmax=max(20,float(desc.pct_2014_degraded.quantile(.95))),
               alpha=.75,edgecolors='none')
axs[1].set(xlabel='Undisturbed forest (%) — 2014',
           ylabel='Regrowth (%) — 2014',
           title='All 216 strips (size = recent clearing)')
axs[1].grid(alpha=.15)
fig.tight_layout()
png=TMF_STRIP_DIR/'WWF_strip_TMF_gradient_overview.png'
fig.savefig(png,dpi=160,bbox_inches='tight')
plt.show()
print('Saved:',png)
print('Note: these are TMF landscape descriptors, not Musanga prevalence or RGB annotation QC.')

### G.2 — Historical disturbance and recovery descriptors (TMF, **as of 2014**)

G.1 quantified the six annual classes for all 216 WWF strips, but **forest-age and disturbance history** are more informative for targeting Musanga pioneer stands.

The JRC provides dedicated **first degradation year** and **first deforestation year** layers, as well as a **2025 endpoint transition map**, intensity and duration. **Do not interpret 2025 endpoint transition subtypes/intensity/duration as conditions visible in WWF imagery acquired in 2014–2015.** This block reconstructs **pre-2015** transitions using annual classes through **December 2014**, plus first-event years with a year ≤2014. We characterize:
- share ever first-degraded / first-deforested by 2014; proportions with first events in **2005–2009** and **2010–2014** (and earlier histories);
- consecutive years in the **degraded (class 2)** and **regrowth (class 4)** states as of end-2014, grouped by **1–3, 4–7 and ≥8 years**;
- historical percentages undisturbed, degraded, deforested and regrowth in **1990, 2000, 2005, 2010 and 2014**;
- **annual TMF class transitions** (1990–2014), distinguished from the number of physical disturbance episodes, and a conservative **recovered from deforested** area measure.

The original G.1 CSV is reused; batch output is cached separately. This is a historical *sampling audit* (not a ground-truth age determination): first-event years are NOT necessarily the latest disturbance date; consecutive-state age is NOT necessarily the exact stand age. TMF uses 30 m pixels, and valid RGB footprint/annotation feasibility must still be checked.

Official TMF sources: <https://forobs.jrc.ec.europa.eu/TMF/data> and <https://forobs.jrc.ec.europa.eu/TMF/resources/tutorial/gee>.

In [ ]:
# G.2 — EXTENDED 2014 HISTORICAL TMF DESCRIPTORS FOR ALL 216 WWF STRIPS
# Runs after G.1; reuses its CSV, writes separately cached batches.
import ee, geopandas as gpd, pandas as pd, numpy as np, matplotlib.pyplot as plt
from shapely.geometry import mapping
from pathlib import Path
from IPython.display import display

EE_PROJECT=(globals().get('EE_PROJECT') or 'musanga-511110').strip()
try:
    ee.Initialize(project=EE_PROJECT)
    ee.Number(1).getInfo()
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)

ROOT_TMFD=WORK_DIR/'TMF_strip_descriptors'
BASE_CSV=ROOT_TMFD/'WWF_all_strips_TMF_2014_2015_descriptors.csv'
assert BASE_CSV.exists(), 'First run G.1, which computes the initial 216-strip TMF composition.'
base=pd.read_csv(BASE_CSV)
assert len(base)==216 and base.plot_id.is_unique
assert 'plots' in globals(), 'Run Section C to restore WWF strip geometries.'

OUT_DIR=ROOT_TMFD/'historical_2014'
OUT_DIR.mkdir(parents=True,exist_ok=True)
BATCH_SIZE_HIST=8
RECOMPUTE_HIST=False
SCALE_HIST_M=30
JRC='projects/JRC/TMF/v1_2025/'
annual=ee.ImageCollection(JRC+'AnnualChanges').mosaic()
first_deg=ee.ImageCollection(JRC+'DegradationYear').mosaic().select(0)
first_def=ee.ImageCollection(JRC+'DeforestationYear').mosaic().select(0)
def year_img(y):
    return annual.select('Dec'+str(y))
a2014=year_img(2014)
area=ee.Image.pixelArea().divide(10000)
metrics=[]
def area_band(mask,name):
    metrics.append(area.multiply(mask.unmask(0)).rename('ha_'+name))
# First event: exclude events AFTER the WWF acquisition.
for layer,short in [(first_deg,'first_degradation'),(first_def,'first_deforestation')]:
    valid=layer.gte(1982).And(layer.lte(2014))
    area_band(valid,short+'_by2014')
    area_band(valid.And(layer.lte(2004)),short+'_through2004')
    area_band(layer.gte(2005).And(layer.lte(2009)),short+'_2005_2009')
    area_band(layer.gte(2010).And(layer.lte(2014)),short+'_2010_2014')
    # Area-weighted mean first-event YEAR (not latest event).
    metrics.append(area.multiply(layer).updateMask(valid).unmask(0)
                   .rename('yeararea_'+short+'_through2014'))
# End-of-2014 current-state consecutive run-lengths: *age in TMF state*, not stand age.
age_reg=ee.Image.constant(0).toInt16()
age_deg=ee.Image.constant(0).toInt16()
transitions=ee.Image.constant(0).toInt16()
prior=year_img(1990)
last_def_pre2014=ee.Image.constant(0).toByte()
for yr in range(1990,2015):
    now=year_img(yr)
    age_reg=age_reg.add(1).multiply(now.eq(4)).toInt16()
    age_deg=age_deg.add(1).multiply(now.eq(2)).toInt16()
    if yr>1990:
        # A change of annual categorical state is not necessarily a distinct disturbance.
        transitions=transitions.add(now.neq(prior)).toInt16()
        last_def_pre2014=last_def_pre2014.Or(prior.eq(3))
    prior=now
for current_age,current_state,short in [(age_reg,4,'regrowth'),(age_deg,2,'degraded')]:
    for lo,hi,label in [(1,3,'1_3y'),(4,7,'4_7y'),(8,25,'8plus_y')]:
        area_band(a2014.eq(current_state).And(current_age.gte(lo))
                  .And(current_age.lte(hi)),short+'_state_age_'+label)
# Pixels in regrowth in 2014 that had at least one year classified deforested before 2014.
area_band(a2014.eq(4).And(last_def_pre2014),
          'regrowth2014_previously_deforested')
for yr in [1990,2000,2005,2010,2014]:
    img=year_img(yr)
    for cls,label in [(1,'undisturbed'),(2,'degraded'),(3,'deforested'),(4,'regrowth')]:
        area_band(img.eq(cls),f'{label}_{yr}')
# Number of 1990–2014 annual state transitions (each categorical transition counts once).
metrics.append(area.multiply(transitions).rename('area_times_transition_count'))
area_band(transitions.gte(1),'any_transition_1990_2014')
area_band(transitions.gte(2),'multiple_transitions_1990_2014')
metric_img=ee.Image.cat(metrics).toFloat()

geo=plots[['plot_id','geometry']].copy().to_crs(4326)
geo=geo[geo.plot_id.isin(base.plot_id)].sort_values('plot_id')
assert len(geo)==216 and geo.plot_id.is_unique
files=[]
for start in range(0,len(geo),BATCH_SIZE_HIST):
    batch=geo.iloc[start:start+BATCH_SIZE_HIST]
    fname=OUT_DIR/f'history_{start:03d}_{start+len(batch):03d}.csv'
    files.append(fname)
    if fname.exists() and not RECOMPUTE_HIST:
        print('Cached',fname.name)
        continue
    features=[ee.Feature(
        ee.Geometry(mapping(rec.geometry),proj='EPSG:4326',geodesic=False),
        {'plot_id':str(rec.plot_id)}) for rec in batch.itertuples(index=False)]
    stats=metric_img.reduceRegions(collection=ee.FeatureCollection(features),
        reducer=ee.Reducer.sum(),scale=SCALE_HIST_M,tileScale=4)
    raw=[f['properties'] for f in stats.getInfo()['features']]
    result=pd.DataFrame(raw)
    assert len(result)==len(batch) and result.plot_id.is_unique
    result.to_csv(fname,index=False)
    print(f'Historical TMF: {start+1}–{start+len(batch)} / {len(geo)}')
hist=pd.concat([pd.read_csv(f) for f in files],ignore_index=True)
assert len(hist)==216 and hist.plot_id.is_unique
merged=base.merge(hist,on='plot_id',how='left',validate='one_to_one')
assert len(merged)==216 and merged['ha_first_deforestation_by2014'].notna().all()
# Convert area metrics to fraction of full survey footprint, not fraction of forest.
for key in hist.columns:
    if key.startswith('ha_'):
        merged['pct_'+key[3:]]=100*pd.to_numeric(merged[key])/merged.footprint_ha
for label in ['first_degradation','first_deforestation']:
    merged['mean_'+label+'_year_by2014']=np.where(
        merged['ha_'+label+'_by2014']>0,
        merged['yeararea_'+label+'_through2014']/merged['ha_'+label+'_by2014'],
        np.nan)
merged['mean_class_transitions_1990_2014']=(
    merged.area_times_transition_count/merged.footprint_ha)
# Conditional proportions within existing degraded or regrowth 2014 classes:
for cat in ['degraded','regrowth']:
    denom=merged[f'ha_{cat}_2014'].replace(0,np.nan)
    for age in ['1_3y','4_7y','8plus_y']:
        key=f'ha_{cat}_state_age_{age}'
        merged[f'pct_within_{cat}_{age}']=100*merged[key]/denom

merged.to_csv(OUT_DIR/'WWF_all_216_strips_TMF_history_asof2014.csv',index=False)
geo_out=geo.merge(merged,on='plot_id',validate='one_to_one')
geo_out.to_file(OUT_DIR/'WWF_all_216_strips_TMF_history_asof2014.gpkg',driver='GPKG')
cols=['plot_id','utm_zone','pct_2014_undisturbed',
      'pct_2014_degraded','pct_2014_regrowth',
      'pct_first_degradation_2010_2014','pct_first_deforestation_2010_2014',
      'pct_regrowth_state_age_1_3y','pct_regrowth_state_age_4_7y',
      'pct_regrowth_state_age_8plus_y','pct_multiple_transitions_1990_2014']
print('Most first-degraded strips in 2010–2014:')
display(merged.sort_values('pct_first_degradation_2010_2014',
                           ascending=False)[cols].head(20).round(2))
print('Most recently deforested strips (first event 2010–2014):')
display(merged.sort_values('pct_first_deforestation_2010_2014',
                           ascending=False)[cols].head(20).round(2))
print('Strips richest in 2014 recent-state regrowth (1–3 yr):')
display(merged.sort_values('pct_regrowth_state_age_1_3y',
                           ascending=False)[cols].head(20).round(2))
print('Distribution across the 216 WWF strips:')
display(merged[[c for c in cols if c.startswith('pct_')]].describe(
    percentiles=[.1,.25,.5,.75,.9]).round(2))

fig,axes=plt.subplots(1,2,figsize=(14,5),constrained_layout=True)
axes[0].scatter(merged.pct_first_degradation_2010_2014,
                merged.pct_first_deforestation_2010_2014,
                s=20+3*merged.pct_2014_regrowth,
                alpha=.7)
axes[0].set(xlabel='First degradation in 2010–2014 (% of strip)',
            ylabel='First deforestation in 2010–2014 (% of strip)',
            title='Historical first-event gradient')
axes[1].scatter(merged.pct_2014_regrowth,
                merged.pct_regrowth_state_age_1_3y,
                s=25,alpha=.7)
axes[1].set(xlabel='All regrowth in 2014 (% of strip)',
            ylabel='Regrowth in 1–3 year consecutive state (% of strip)',
            title='Young TMF regrowth vs total regrowth')
for ax in axes:ax.grid(alpha=.15)
fig_path=OUT_DIR/'WWF_TM F_historical_gradient_2014.png'.replace(' ','')
fig.savefig(fig_path,dpi=160)
plt.show()
print('Saved history to',OUT_DIR)
print('Caution: 2025 TransitionMap_Subtypes / duration / intensity are not 2014 classes.')


## H — TMF-guided sampling of new WWF exhaustive annotation circles

**Research aim:** expand Musanga labels to secondary and regenerating forest, degraded canopy, and recently cleared landscapes, plus undisturbed reference forest. The WWF RGB imagery is the annotation source; TMF is a **coarse (30 m), contextual sampling stratifier**, not a Musanga label or a claim about individual 125 m² circles. <https://forobs.jrc.ec.europa.eu/TMF/resources/tutorial/gee>

**Temporal control:** The WWF survey imagery was acquired during **2014–2015** (individual plot years not yet distinguished). Sample only TMF states that remain consistent across *both* 2014 and 2015; recently cleared candidates must already be cleared by 2014. Later TMF classes are never used as contemporary labels. "Recently cleared" means transition from forest to deforested land within the preceding three years, as measured in the relevant TMF annual maps. "Regrowth" is the official TMF regrowth state at the acquisition year.

**Confirmed annotation unit:** 125 m **diameter** (radius 62.5 m; area approximately 1.227 ha). Candidate circles and exports use this diameter by default.

**Workflow:** verify dates → sample TMF strata within actual WWF LiDAR footprints → inspect 250 m context + native RGB validity → export selected circles and small georeferenced RGB clips to Drive → hand annotation in GIS → independent label QC. No automatic Musanga annotations or giant mosaic downloads.

In [ ]:
# H1 — Temporal metadata checklist and explicit, configurable annotation area
import math, pandas as pd
from pathlib import Path
TMF_OUT=WORK_DIR/'TMF_guided_annotation'
TMF_OUT.mkdir(parents=True,exist_ok=True)
PILOT_PLOTS=['UTM33S_Plot135','UTM34N_Plot105','UTM34S_Plot60',
             'UTM35N_Plot100','UTM35S_Plot79','UTM36N_Plot104']
CIRCLE_MODE='diameter_m'  # confirmed: 125 m diameter, not 125 m²
CIRCLE_VALUE=125.0
CIRCLE_RADIUS_M=(math.sqrt(CIRCLE_VALUE/math.pi)
                 if CIRCLE_MODE=='area_m2' else CIRCLE_VALUE/2)
print('Circle mode:',CIRCLE_MODE,'radius (m):',round(CIRCLE_RADIUS_M,2),
      'diameter (m):',round(2*CIRCLE_RADIUS_M,2),
      'area (m²):',round(math.pi*CIRCLE_RADIUS_M**2,1))
# User-confirmed WWF campaign dates: 2014–2015, individual plot years unknown.
# H2 will sample only candidate landscape states stable across BOTH years.
WWF_RGB_ACQUISITION_WINDOW=(2014,2015)
WWF_RGB_ACQUISITION_YEAR={p:None for p in PILOT_PLOTS}
print('Campaign period:',WWF_RGB_ACQUISITION_WINDOW,
      '| individual plot years remain unverified; use stable-class sampling.')
# List potentially useful bucket reports and metadata filenames (metadata requests only)
helpful=df[df.path.str.contains(r'00-raw-data/Reports/|\.xml$|\.pdf$',case=False,regex=True)]
display(helpful[['path','size_GB']].head(55))
print('Note: filename/2016 national products do NOT establish individual RGB flight year.')

### H.1B — Discover original flight dates before TMF selection
Inspect RGB GeoTIFF tags and list associated metadata/reports. **Do not mistake a file update, TIFF processing timestamp or 2016 map product date for an aerial acquisition date.** This is evidence gathering: candidate years must be verified before H.2.

In [ ]:
# H1B — Discover WWF flight acquisition dates; metadata only, no full imagery download
import re, json, pandas as pd, rasterio
from pathlib import PurePosixPath
from IPython.display import display
# Recognize acquisition-specific metadata fields; generic TIFF DateTime is often processing time.
date_key=re.compile(r'acqui|capture|flight|survey|sensing|exposure|gps.*date|date.*gps',re.I)
year_re=re.compile(r'(?<!\d)(19\d{2}|20\d{2})(?!\d)')
date_rows=[]
for plot_id in PILOT_PLOTS:
    rgb=f'001_Lidar_Rasters/RGB_mosaic/{plot_id}_RGB.tif'
    hits=[]
    try:
        with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR'):
            with rasterio.open('/vsigs/'+BUCKET+'/'+rgb) as ds:
                namespaces=[''] + [k for k in ds.tag_namespaces() if k]
                for ns in dict.fromkeys(namespaces):
                    try:
                        tags=ds.tags(ns=ns) if ns else ds.tags()
                    except Exception:
                        continue
                    for key,value in tags.items():
                        if date_key.search(key):
                            hits.append(dict(source='geotiff_tag',key=f'{ns}:{key}',value=str(value)[:300]))
                        elif re.search(r'datetime|software',key,re.I):
                            hits.append(dict(source='processing_tag_UNVERIFIED',key=f'{ns}:{key}',value=str(value)[:300]))
    except Exception as exc:
        hits.append(dict(source='ERROR',key='read',value=repr(exc)[:300]))
    # Candidate supporting filenames from source raw plot and report directories
    raw_files=df[df.path.str.contains(re.escape(plot_id.split('_')[0]),regex=True,na=False) &
                 df.path.str.contains(r'\.xml$|\.pdf$|\.txt$|\.csv$',case=False,regex=True,na=False)]
    for filename in raw_files.path.head(10):
        hits.append(dict(source='supporting_filename_ONLY',key='object',value=filename))
    if not hits:
        hits=[dict(source='NO_ACQUISITION_DATE_FOUND',key='',value='')]
    for hit in hits:
        date_rows.append(dict(plot_id=plot_id,**hit))
date_evidence=pd.DataFrame(date_rows)
display(date_evidence)
date_evidence.to_csv(TMF_OUT/'wwf_acquisition_date_evidence.csv',index=False)
# Reports inventory is small; inspect original survey reports if tags don't contain actual flight dates.
reports=df[df.path.str.startswith('00-raw-data/Reports/') & df.path.str.endswith('.pdf')]
print('\nSurvey reports (potential flight dates):')
display(reports[['path','size_GB']].head(70))
print('Evidence exported:',TMF_OUT/'wwf_acquisition_date_evidence.csv')
print('Fill WWF_RGB_ACQUISITION_YEAR in H1 only when a genuine flight acquisition year is established.')


In [ ]:
# H2 — Conservative TMF sampling for WWF RGB aerial survey 2014–2015
# Only sites with matching TMF annual state in Dec2014 and Dec2015 qualify.
# 'Recently cleared' must already be deforested in 2014 and 2015,
# having been forest in 2011 or 2012. No post-2015 forest conditions used.
import ee, geopandas as gpd, numpy as np, pandas as pd, rasterio
from shapely.geometry import Point, mapping
from shapely.ops import transform as shape_transform
from pyproj import Transformer
assert tuple(WWF_RGB_ACQUISITION_WINDOW)==(2014,2015)
# Self-healing setup: GCS authentication alone does NOT initialize Earth Engine.
EE_PROJECT=(globals().get('EE_PROJECT') or 'musanga-511110').strip()
assert EE_PROJECT, 'EE_PROJECT must not be empty'
print('GEE project:',repr(EE_PROJECT))
try:
    ee.Initialize(project=EE_PROJECT)
    print('Earth Engine connected:',EE_PROJECT,'| test:',ee.Number(1).getInfo())
except Exception as first_error:
    print('Earth Engine setup needed:',str(first_error)[:300])
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)
    print('Earth Engine authenticated:',EE_PROJECT,'| test:',ee.Number(1).getInfo())
tmf=ee.ImageCollection('projects/JRC/TMF/v1_2025/AnnualChanges').mosaic()
annual14=tmf.select('Dec2014')
annual15=tmf.select('Dec2015')
forest11=tmf.select('Dec2011').eq(1).Or(tmf.select('Dec2011').eq(2))
forest12=tmf.select('Dec2012').eq(1).Or(tmf.select('Dec2012').eq(2))
# Stable class 1,2,4; class 3 only if recently deforested before image capture.
recent_clear=annual14.eq(3).And(annual15.eq(3)).And(forest11.Or(forest12))
stable=annual14.eq(annual15)
stratum=ee.Image.constant(0).where(stable.And(annual14.eq(1)),1)
stratum=stratum.where(stable.And(annual14.eq(2)),2)
stratum=stratum.where(recent_clear,3)
stratum=stratum.where(stable.And(annual14.eq(4)),4).rename('stratum').toByte()
# One 30 m TMF pixel is 900 m²: candidate forest category is broad spatial context,
# NOT a verified label for the much smaller annotation circle.
SAMPLES_PER_STRATUM_PER_PLOT=5
KEEP_ONLY_IF_WITHIN_PLOT=True
rows=[]
crs_by_plot={}
for plot_id in PILOT_PLOTS:
    plotrow=plots.loc[plots.plot_id==plot_id]
    assert len(plotrow)==1, plot_id
    geom=plotrow.geometry.iloc[0]
    region=ee.Geometry(mapping(geom),proj='EPSG:4326',geodesic=False)
    fc=stratum.updateMask(stratum.gt(0)).stratifiedSample(
        numPoints=SAMPLES_PER_STRATUM_PER_PLOT,classBand='stratum',
        classValues=[1,2,3,4],
        classPoints=[SAMPLES_PER_STRATUM_PER_PLOT]*4,
        region=region,scale=30,geometries=True,seed=8108,
        dropNulls=True,tileScale=4)
    features=fc.getInfo()['features']
    with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR'):
        with rasterio.open('/vsigs/'+BUCKET+
             f'/001_Lidar_Rasters/RGB_mosaic/{plot_id}_RGB.tif') as src:
            assert src.crs and src.crs.is_projected and src.crs.linear_units=='metre'
            crs_by_plot[plot_id]=src.crs
    for f in features:
        lon,lat=f['geometry']['coordinates']
        cls=int(f['properties']['stratum'])
        rows.append(dict(plot_id=plot_id,TMF_years='2014;2015',
                         time_policy='stable_in_both_2014_and_2015',
                         tmf_stratum=cls,
                         stratum_label={1:'undisturbed',2:'degraded',
                           3:'cleared_by_2014_after_2011_2012_forest',
                           4:'regrowth'}[cls],
                         longitude=float(lon),latitude=float(lat)))
    print(plot_id,':',len(features),'TMF candidate centres')
assert rows,'No eligible TMF candidate centres found'
table=pd.DataFrame(rows)
proposals=[]
for i,r in enumerate(table.itertuples(index=False),1):
    local_crs=crs_by_plot[r.plot_id]
    proj=Transformer.from_crs('EPSG:4326',local_crs,always_xy=True)
    x,y=proj.transform(r.longitude,r.latitude)
    circle=Point(x,y).buffer(CIRCLE_RADIUS_M,resolution=32)
    footprint=gpd.GeoSeries(
        [plots.loc[plots.plot_id==r.plot_id,'geometry'].iloc[0]],
        crs='EPSG:4326').to_crs(local_crs).iloc[0]
    if not footprint.covers(circle):continue
    rev=Transformer.from_crs(local_crs,'EPSG:4326',always_xy=True)
    proposals.append(dict(candidate_id=f'WWF_{i:04d}',plot_id=r.plot_id,
        TMF_years=r.TMF_years,time_policy=r.time_policy,
        tmf_stratum=r.tmf_stratum,stratum_label=r.stratum_label,
        geometry=shape_transform(rev.transform,circle)))
assert proposals,'Candidates exist but circles did not fit in mapped WWF footprints'
gdf=gpd.GeoDataFrame(proposals,geometry='geometry',crs='EPSG:4326')
gdf['rgb_validated']=False
gdf['visually_approved']=False
gdf['musanga_annotation_status']='NOT_ANNOTATED'
gdf.to_file(TMF_OUT/'wwf_tmf_candidate_circles.gpkg',driver='GPKG')
gdf.drop(columns='geometry').to_csv(TMF_OUT/'wwf_tmf_candidate_circles.csv',index=False)
print('Proposed (not visually approved) circles:',len(gdf))
display(gdf.groupby(['plot_id','stratum_label']).size().rename('n').reset_index())
print('TMF classes must be confirmed visually in 2014–2015 RGB.')


In [ ]:
# H3 — Export bounded, native RGB context chips for visual review
# Select at most one candidate per plot × TMF class initially, no bulk download.
# Images contain full 250 m context; annotation circle stays much smaller.
import rasterio, numpy as np, geopandas as gpd, pandas as pd
from rasterio.windows import from_bounds, Window
from rasterio.windows import transform as window_transform
from rasterio.enums import Resampling
from pathlib import Path
CHIP_CONTEXT_M=250
MAX_CHIPS=12
RUN_CHIP_EXPORT=False  # change to True after inspecting candidate table
CHIP_DIR=TMF_OUT/'rgb_native_context_chips'
CHIP_DIR.mkdir(parents=True,exist_ok=True)
if RUN_CHIP_EXPORT:
    selection=gdf.sort_values(['plot_id','tmf_stratum','candidate_id']).groupby(
        ['plot_id','tmf_stratum']).head(1).head(MAX_CHIPS)
    chip_logs=[]
    for row in selection.itertuples(index=False):
        uri='/vsigs/'+BUCKET+f'/001_Lidar_Rasters/RGB_mosaic/{row.plot_id}_RGB.tif'
        try:
            with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR'):
                with rasterio.open(uri) as src:
                    circle=gpd.GeoSeries([row.geometry],crs='EPSG:4326').to_crs(src.crs).iloc[0]
                    x,y=circle.centroid.x,circle.centroid.y
                    half=CHIP_CONTEXT_M/2
                    win=from_bounds(x-half,y-half,x+half,y+half,src.transform)
                    win=win.round_offsets().round_lengths()
                    full=Window(0,0,src.width,src.height)
                    win=win.intersection(full)
                    a=src.read(indexes=[1,2,3],window=win,masked=True)
                    data=np.ma.filled(a,0)
                    valid=(~np.ma.getmaskarray(a).any(axis=0)) & np.any(data!=0,axis=0)
                    fraction=float(valid.mean())
                    if fraction<.70:
                        print('SKIP low-validity',row.candidate_id,round(fraction,3))
                        continue
                    profile=src.profile.copy()
                    profile.update(driver='GTiff',count=3,width=int(win.width),
                                   height=int(win.height),transform=window_transform(win,src.transform),
                                   tiled=True,compress='deflate',BIGTIFF='IF_SAFER')
                    for key in ['photometric','interleave']:
                        profile.pop(key,None)
                    dst=CHIP_DIR/f'{row.candidate_id}_{row.plot_id}_{row.stratum_label}.tif'
                    with rasterio.open(dst,'w',**profile) as sink:
                        sink.write(data)
                        sink.colorinterp=src.colorinterp[:3]
                    chip_logs.append(dict(candidate_id=row.candidate_id,
                                          plot_id=row.plot_id,context_m=CHIP_CONTEXT_M,
                                          native_gsd_cm=100*abs(src.res[0]),
                                          valid_fraction=fraction,rgb_tif=str(dst)))
                    print('Saved:',dst.name,'| valid',round(fraction,3))
        except Exception as ex:
            print('FAILED:',row.candidate_id,repr(ex))
    pd.DataFrame(chip_logs).to_csv(TMF_OUT/'wwf_context_chip_exports.csv',index=False)
    print('Use GeoPackage circles as separate QGIS overlay. Do NOT confuse TMF stratum with a field label.')
else:
    print('Export OFF. Review H2 candidates, then set RUN_CHIP_EXPORT=True.')
print('Working directory:',TMF_OUT)

### H.4 — Human exhaustive annotation and quality controls

Load `wwf_tmf_candidate_circles.gpkg` and corresponding **georeferenced** RGB chips into QGIS. First check the imagery acquisition year, validity, and apparent disturbance/regrowth stage; reject mismatched areas. Annotate all visible Musanga crowns inside each approved circle and explicitly record true absence as zero. Keep uncertainty labels for ambiguous young crowns; do **not** treat unknown pixels as negative. Record annotator, date, annotation confidence, tree/crown counts, and image quality. For a robust cross-forest validation exercise, group spatially nearby circles by parent WWF plot to avoid leakage.

Sampling reliability: TMF pixels are ~30 m, coarser than the crown-scale detail within a **125 m diameter** circle. TMF determines the surrounding landscape class, not the circle's actual vegetation. Candidate status is never ground truth. Crop valid coverage and human photo interpretation must confirm suitability before annotation.

**Campaign date 2014–2015:** H2 uses only TMF candidate states stable across these two years. Verify per-plot dates later for finer transition sampling.

## H.5 — Shareable Earth Engine assets for annotation circles

Publish a deterministic 12-circle pilot (three per TMF category, diverse WWF plots) or all 90 circles into `musanga-511110`. Two **separate** table assets are created: the **125 m diameter exhaustive annotation circles** and the **225 m diameter context boundaries**, representing an additional 50 m buffer. Circles are buffered in their own metric UTM CRS before conversion to WGS84. Assets preserve circle identifiers, site, TMF sampling category, and unreviewed annotation status. This operation **does not upload WWF RGB imagery or create editable annotations**; imagery must be separately ingested as EE image assets (subject to redistribution permissions), and annotations should be saved in a new dedicated vector layer.

The cell starts Earth Engine export tasks. Wait for both tasks to complete before using their IDs in the <https://code.earthengine.google.com/> Code Editor. Share the table assets with collaborators using Earth Engine Assets permissions.

In [ ]:
# H.5 — Publish WWF pilot annotation circles + 50 m context buffers to GEE
# This creates TWO table assets: 125 m diameter cores and 225 m context circles.
# Default: same 12 reproducible examples as H.2B (3 per TMF class).
# To publish all candidate circles, change EXPORT_SCOPE = 'all_90'.
import re, ee, pandas as pd, geopandas as gpd
from shapely.geometry import mapping

EE_PROJECT = (globals().get('EE_PROJECT') or 'musanga-511110').strip()
ee.Initialize(project=EE_PROJECT)

EXPORT_SCOPE = 'pilot_12'     # 'pilot_12' or 'all_90'
N_PER_CLASS = 3
SELECTION_SEED = 42
OUTER_BUFFER_M = 50.0        # additional 50 m outside the 62.5 m core
SUBMIT_EXPORT_TASKS = True   # set False to preview only

if EXPORT_SCOPE == 'pilot_12':
    labels = ['undisturbed','degraded',
              'cleared_by_2014_after_2011_2012_forest','regrowth']
    selections=[]
    for label in labels:
        sub=gdf[gdf['stratum_label'].eq(label)].sample(frac=1,random_state=SELECTION_SEED)
        distinct=sub.drop_duplicates('plot_id').head(N_PER_CLASS)
        remainder=sub.loc[~sub.index.isin(distinct.index)].head(N_PER_CLASS-len(distinct))
        selections.append(pd.concat([distinct,remainder]))
    export_gdf=gpd.GeoDataFrame(pd.concat(selections,ignore_index=True),
                                geometry='geometry',crs=gdf.crs)
elif EXPORT_SCOPE == 'all_90':
    export_gdf=gdf.copy()
else:
    raise ValueError('EXPORT_SCOPE must be pilot_12 or all_90')

assert len(export_gdf)>0
assert export_gdf.candidate_id.is_unique, 'Duplicate candidate IDs'
assert export_gdf.crs.to_epsg()==4326, 'Expected WGS84 original circles'
# Fail if H1 was accidentally run in 125 m² mode.
assert abs(CIRCLE_RADIUS_M-62.5)<0.01, 'Expected 125 m DIAMETER (radius 62.5 m)'

def plot_utm_epsg(plot_id):
    found=re.match(r'^UTM(\d{1,2})([NS])_',str(plot_id))
    if not found:raise ValueError('Unexpected plot UTM zone: '+str(plot_id))
    zone=int(found.group(1))
    return (32600 if found.group(2)=='N' else 32700)+zone

core_features=[]
context_features=[]
for _,row in export_gdf.iterrows():
    # Geometries are made in a local metric CRS (one of several WWF UTM zones).
    utm=plot_utm_epsg(row['plot_id'])
    circle=gpd.GeoSeries([row.geometry],crs='EPSG:4326').to_crs(epsg=utm).iloc[0]
    expanded=circle.buffer(OUTER_BUFFER_M,resolution=48)
    center=circle.centroid
    assert abs(circle.area - 3.141592653589793*62.5**2)<35, (
        'Core does not have the expected 125 m diameter: '+str(row['candidate_id']))
    geom_core=gpd.GeoSeries([circle],crs=f'EPSG:{utm}').to_crs(4326).iloc[0]
    geom_context=gpd.GeoSeries([expanded],crs=f'EPSG:{utm}').to_crs(4326).iloc[0]
    attrs={
        'circle_id':str(row['candidate_id']),
        'plot_id':str(row['plot_id']),
        'tmf_class':str(row['stratum_label']),
        'tmf_code':int(row['tmf_stratum']),
        'image_years':'2014-2015',
        'sampling':'TMF_2014_2015_stable',
        'review':'NOT_REVIEWED',
        'annotation':'NOT_ANNOTATED',
        'core_diam_m':125,
        'context_diam_m':225,
    }
    core_features.append(ee.Feature(
        ee.Geometry(mapping(geom_core),geodesic=False),
        {**attrs,'boundary':'core_125m'}))
    context_features.append(ee.Feature(
        ee.Geometry(mapping(geom_context),geodesic=False),
        {**attrs,'boundary':'context_225m'}))

core_fc=ee.FeatureCollection(core_features)
context_fc=ee.FeatureCollection(context_features)
suffix='pilot12' if EXPORT_SCOPE=='pilot_12' else 'all90'
core_asset=f'projects/{EE_PROJECT}/assets/WWF_TMF_2014_2015_{suffix}_core_125m'
context_asset=f'projects/{EE_PROJECT}/assets/WWF_TMF_2014_2015_{suffix}_context_225m'

print('Circles selected:',len(export_gdf))
display(export_gdf[['candidate_id','plot_id','stratum_label']])
print('Core asset:',core_asset)
print('Context asset:',context_asset)

# Preserve the same candidates locally for annotation logistics.
export_gdf.to_file(
    TMF_OUT/f'WWF_TMF_2014_2015_{suffix}_core125m.gpkg',driver='GPKG')

if SUBMIT_EXPORT_TASKS:
    for title,fc,asset in [
        ('core125m',core_fc,core_asset),
        ('context225m',context_fc,context_asset),
    ]:
        try:
            existing=ee.data.getAsset(asset)
        except ee.EEException as err:
            if 'not found' in str(err).lower() or '404' in str(err):
                existing=None
            else:
                raise
        if existing is not None:
            print('ALREADY EXISTS (not overwritten):',asset)
            continue
        task=ee.batch.Export.table.toAsset(
            collection=fc,
            description=f'WWF_TMF_{suffix}_{title}',
            assetId=asset)
        task.start()
        print('Submitted export:',title,'task ID:',task.id)
    print('Check the Earth Engine Tasks tab. An export is not immediately available.')
else:
    print('Dry run only; set SUBMIT_EXPORT_TASKS=True to create EE assets.')
